# 04 · 站在巨人肩上:stable-baselines3

手刻 DQN 讓你懂了原理,但真正做專案時,沒人會每次重寫訓練迴圈。**stable-baselines3 (SB3)** 把 DQN、PPO、A2C 等演算法封裝成幾行就能用、又經過嚴格調校的實作。

這一課:用 SB3 在 CartPole 上訓練,**一行 `model.learn()`** 取代上一課整個迴圈;並看 TensorBoard 監控訓練。

## 1. 安裝 stable-baselines3

In [ ]:
%pip install -q -U "stable-baselines3>=2.0" "gymnasium[classic-control]"

## 2. 用 PPO 訓練 CartPole

PPO(Proximal Policy Optimization)是目前最常用的通用演算法,穩、好調。建立模型 → `learn()` → 完成,就這樣。

In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO

env = gym.make("CartPole-v1")
model = PPO("MlpPolicy", env, verbose=0, tensorboard_log="./tb_cartpole")
model.learn(total_timesteps=30_000)      # 上一課整個迴圈,濃縮成這一行
print("訓練完成。")

## 3. 評估:跑 20 回合看平均撐多久

CartPole-v1 滿分 500,學成後應該常常頂到上限。

In [ ]:
import numpy as np

scores = []
for _ in range(20):
    obs, _ = env.reset()
    done, total = False, 0.0
    while not done:
        action, _ = model.predict(obs, deterministic=True)   # 用學到的策略
        obs, r, terminated, truncated, _ = env.step(int(action))
        done = terminated or truncated
        total += r
    scores.append(total)
print(f"平均得分:{np.mean(scores):.1f}  (滿分 500)")

## 4. 換一行就換演算法

SB3 的價值:介面統一。想試 DQN?把 `PPO` 換成 `DQN` 就好,其餘不動。

In [ ]:
from stable_baselines3 import DQN

dqn = DQN("MlpPolicy", gym.make("CartPole-v1"), verbose=0)
dqn.learn(total_timesteps=30_000)
print("DQN 也訓練完成——同一套介面,換演算法只改一個名字。")

## 5. TensorBoard:看訓練過程

訓練時 SB3 已把指標寫進 `./tb_cartpole`。在 Colab 用下面兩行就能看互動圖表(獎勵、loss、explained variance…)。

In [ ]:
# 在 Colab 執行這兩行,會內嵌 TensorBoard 面板
%load_ext tensorboard
%tensorboard --logdir ./tb_cartpole

## 小結

- **SB3 把演算法封裝好**:建立模型 → `learn()` → `predict()`,訓練迴圈不用自己寫。
- **介面統一**:PPO / DQN / A2C 換一個名字就換演算法。
- **TensorBoard** 是 RL 的儀表板,訓練不順時先看它。
- 手刻(懂原理)+ SB3(做專案),兩個都要會。

下一課:回到手刻,認識另一大家族——**策略梯度**,直接學「策略」而不是學 Q 值。